# Analysis

Reads per-seed JSONs from `results/` and produces:

1. Headline structural table (4 cells × 2 datasets) with paired-t p-values for the four named contrasts.
2. Text on/off lift table (3 models × 2 datasets) with paired-t p-values.
3. Cascade-size bucket F1 (<100 / 100–500 / >500) for the headline contrast.
4. Attention heatmaps for 5–10 cascades (2–3 per class) + per-cascade entropy and distance-from-root stats.

In [ ]:
import sys
sys.path.insert(0, "..")

import glob
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import numpy as np
import torch

from analysis.stats import load_run_files, group_runs, summarise, paired_t

RESULTS_DIR = "../results"
FIG_DIR = Path("../results/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS = ("gcn", "gat", "bigcn", "gps", "cascade_gps")
DATASETS = ("twitter15", "twitter16")

runs = load_run_files(RESULTS_DIR)
groups = group_runs(runs)
print(f"loaded {len(runs)} runs across {len(groups)} (model, dataset, text, readout) cells")
for k, v in sorted(groups.items()):
    print(f"  {k} : {len(v)} seeds")

## Headline structural table (`use_text=False`)

Cells: GCN-mean, GAT-mean, GraphGPS-mean, GraphGPS-4way. Per-dataset mean ± std test macro-F1 over 5 seeds.

In [ ]:
STRUCT_CELLS = [
    ("gcn", "mean"),
    ("gat", "mean"),
    ("bigcn", "mean"),
    ("gps", "mean"),
    ("gps", "4way"),
    ("cascade_gps", "mean"),
]
rows = []
for model, readout in STRUCT_CELLS:
    for dataset in DATASETS:
        key = (model, dataset, False, readout)
        if key not in groups:
            rows.append((model, dataset, readout, None))
            continue
        s = summarise(groups[key])
        rows.append((model, dataset, readout, s))

print(f"{'Model':<10} {'Readout':<8} {'Dataset':<12} {'Test F1 (mean ± std)':<24} {'n':>3}")
print("-" * 60)
for model, dataset, readout, s in rows:
    if s is None:
        print(f"{model:<10} {readout:<8} {dataset:<12} (no results)")
    else:
        print(f"{model:<10} {readout:<8} {dataset:<12} {s['mean']:.3f} ± {s['std']:.3f}{'':<14} {s['n']:>3}")

In [ ]:
# Four structural contrasts: GAT − GCN, GPS(mean) − GAT, GPS(4way) − GPS(mean), GPS(4way) − GAT
CONTRASTS = [
    ("GAT − GCN",              ("gat", "mean"), ("gcn", "mean")),
    ("GPS(mean) − GAT",        ("gps", "mean"), ("gat", "mean")),
    ("GPS(4way) − GPS(mean)",  ("gps", "4way"), ("gps", "mean")),
    ("GPS(4way) − GAT",        ("gps", "4way"), ("gat", "mean")),
]

print(f"{'Contrast':<26} {'Dataset':<12} {'Δ F1':>8} {'std':>7} {'p (paired)':>12}")
print("-" * 70)
for name, (ma, ra), (mb, rb) in CONTRASTS:
    for dataset in DATASETS:
        a = groups.get((ma, dataset, False, ra))
        b = groups.get((mb, dataset, False, rb))
        if not a or not b:
            print(f"{name:<26} {dataset:<12}   (missing runs)")
            continue
        t = paired_t(a, b)
        sig = " *" if t["p"] < 0.05 else ""
        print(f"{name:<26} {dataset:<12} {t['mean_delta']:+.3f}  {t['std_delta']:.3f}  {t['p']:.4f}{sig}")

## Text lift table (text-on vs text-off, paired per seed)

For each (model, dataset) at the structural winner readout, computes Δ F1 = test-F1(text=on) − test-F1(text=off), paired by seed at the structural winner readout.

In [ ]:
def winner_readout(model: str, dataset: str) -> str | None:
    """Return the structural readout with the highest mean val-F1; None if no runs exist."""
    candidates = [k for k in groups if k[0] == model and k[1] == dataset and k[2] is False]
    if not candidates:
        return None
    return max(candidates, key=lambda k: summarise(groups[k], metric="val_f1")["mean"])[3]

print(f"{'Model':<10} {'Dataset':<12} {'Readout':<8} {'F1 off':>8} {'F1 on':>8} {'Δ F1':>8} {'p (paired)':>12}")
print("-" * 75)
for model in MODELS:
    for dataset in DATASETS:
        readout = winner_readout(model, dataset)
        if readout is None:
            print(f"{model:<10} {dataset:<12} (no structural runs)")
            continue
        off = groups.get((model, dataset, False, readout))
        on  = groups.get((model, dataset, True, readout))
        if not off or not on:
            print(f"{model:<10} {dataset:<12} {readout:<8}  (text-on missing)")
            continue
        s_off = summarise(off); s_on = summarise(on)
        t = paired_t(on, off)
        sig = " *" if t['p'] < 0.05 else ""
        print(f"{model:<10} {dataset:<12} {readout:<8} {s_off['mean']:.3f}    {s_on['mean']:.3f}    {t['mean_delta']:+.3f}    {t['p']:.4f}{sig}")

## Cascade size buckets for the headline contrast

Loads GPS(4way) `use_text=False` checkpoints and re-evaluates per cascade-size bucket: <100 / 100–500 / >500 nodes.

Since trainer doesn't save checkpoints, we re-derive per-cascade predictions from the saved confusion matrices? No — confusion matrices are aggregated. The proper way: re-run inference with the same seeds. **This cell is a stub** that documents the analysis hook; populate after re-running with per-cascade prediction dumps if needed for the final report.

In [ ]:
from data.dataset import CascadeDataset
from training.trainer import stratified_split

ds15 = CascadeDataset(root="../Twitter15_Twitter16", name="twitter15")
ds16 = CascadeDataset(root="../Twitter15_Twitter16", name="twitter16")

BUCKET_EDGES = (100, 500)
def bucket(n: int) -> str:
    if n < BUCKET_EDGES[0]: return "<100"
    if n <= BUCKET_EDGES[1]: return "100-500"
    return ">500"

for ds, name in ((ds15, "twitter15"), (ds16, "twitter16")):
    _, _, test_idx = stratified_split(ds, split_seed=0)
    sizes = [ds[i].num_nodes for i in test_idx]
    counts = {b: 0 for b in ("<100", "100-500", ">500")}
    for s in sizes:
        counts[bucket(s)] += 1
    print(f"{name} test set:  <100={counts['<100']}  100-500={counts['100-500']}  >500={counts['>500']}  (total={len(sizes)})")

print("\nTo report per-bucket F1, dump per-cascade predictions during training (extend trainer eval) and bucket here.")

## Attention visualization (GraphGPS, layer 1)

Pick 5–10 test-set cascades, 2–3 per class, deterministically by sorted cascade ID. Extract layer-1 global-attention probabilities from a trained GraphGPS checkpoint and plot a heatmap per cascade. Report per-cascade entropy and the mean distance-from-root of the top-3 attended nodes.

**Prereq**: a saved GraphGPS(4way) checkpoint at `../results/gps_twitter15_text-off_readout-4way_seed0.pt`. The trainer writes one per seed when `results_dir` is set.

In [ ]:
import math
from collections import defaultdict

from torch_geometric.utils import to_dense_batch

from models.gps import GPSClassifier
from training.trainer import stratified_split
from data.dataset import LABEL_MAP

INV_LABEL = {v: k for k, v in LABEL_MAP.items()}
DATASET = ds15  # change to ds16 for the other dataset
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

_, _, test_idx = stratified_split(DATASET, split_seed=0)

# Deterministic 2 cascades per class by sorted dataset index.
PER_CLASS = 2
selected_idx: list[int] = []
by_class: dict[int, list[int]] = defaultdict(list)
for i in sorted(test_idx):
    by_class[int(DATASET[i].y.item())].append(i)
for c in sorted(by_class):
    selected_idx.extend(by_class[c][:PER_CLASS])
print("selected cascades:", [(i, INV_LABEL[int(DATASET[i].y.item())], DATASET[i].num_nodes) for i in selected_idx])

GPS_KWARGS = dict(in_channels=DATASET[0].x.shape[1], hidden_channels=128, num_layers=3,
                  heads=4, dropout=0.1, edge_dim=DATASET[0].edge_attr.shape[1], readout="4way")
model = GPSClassifier(**GPS_KWARGS).to(DEVICE)
ckpt = Path("../results/gps_twitter15_text-off_readout-4way_seed0.pt")
if not ckpt.exists():
    raise FileNotFoundError(f"checkpoint not found: {ckpt}. Train GraphGPS(4way) seed 0 first.")
model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
model.eval()
print(f"loaded checkpoint: {ckpt}")

def layer1_attn_probs(data) -> np.ndarray:
    """Return [N, N] head-averaged attention probabilities for the cascade, layer 1 (convs[0])."""
    data = data.to(DEVICE)
    with torch.no_grad():
        x = model.input_proj(data.x)
        conv0 = model.convs[0]
        batch = torch.zeros(x.size(0), dtype=torch.long, device=DEVICE)
        dense, mask = to_dense_batch(x, batch)
        _, w = conv0.attn(dense, dense, dense, key_padding_mask=~mask,
                          need_weights=True, average_attn_weights=True)
    return w[0].cpu().numpy()

rows = []
for idx in selected_idx:
    data = DATASET[idx]
    A = layer1_attn_probs(data)
    p_root = A[0]
    p_root = p_root / max(p_root.sum(), 1e-12)
    entropy = float(-np.sum(p_root * np.log(np.clip(p_root, 1e-12, None))))
    depths = data.x[:, 1].numpy()  # feature col 1 is depth (normalized)
    top3 = np.argsort(-p_root)[:3]
    rows.append({
        "idx": idx, "label": INV_LABEL[int(data.y.item())], "N": int(data.num_nodes),
        "entropy": entropy, "top3_depth_norm": depths[top3].tolist(),
    })

    fig, ax = plt.subplots(figsize=(4, 3.5))
    im = ax.imshow(A, aspect="auto", cmap="viridis")
    ax.set_title(f"idx={idx} N={data.num_nodes} {INV_LABEL[int(data.y.item())]}")
    ax.set_xlabel("key node"); ax.set_ylabel("query node")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    out = FIG_DIR / f"attn_layer1_idx{idx}.png"
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {out}")

print("\nper-cascade summary:")
for r in rows:
    print(r)


## Final tables and figures

Saves the headline + text-lift tables to CSV under `results/` so they can be pasted into the write-up.

In [ ]:
import csv

out_headline = Path("../results/table_headline.csv")
with open(out_headline, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["model", "readout", "dataset", "test_f1_mean", "test_f1_std", "n_seeds"])
    for model, readout in STRUCT_CELLS:
        for dataset in DATASETS:
            g = groups.get((model, dataset, False, readout))
            if not g:
                w.writerow([model, readout, dataset, "", "", 0])
                continue
            s = summarise(g)
            w.writerow([model, readout, dataset, f"{s['mean']:.4f}", f"{s['std']:.4f}", s['n']])
print(f"wrote {out_headline}")

out_contrasts = Path("../results/table_contrasts.csv")
with open(out_contrasts, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["contrast", "dataset", "mean_delta", "std_delta", "t", "p", "n_pairs"])
    for name, (ma, ra), (mb, rb) in CONTRASTS:
        for dataset in DATASETS:
            a = groups.get((ma, dataset, False, ra))
            b = groups.get((mb, dataset, False, rb))
            if not a or not b:
                w.writerow([name, dataset, "", "", "", "", 0]); continue
            t = paired_t(a, b)
            w.writerow([name, dataset, f"{t['mean_delta']:.4f}", f"{t['std_delta']:.4f}", f"{t['t']:.4f}", f"{t['p']:.4f}", t['n_pairs']])
print(f"wrote {out_contrasts}")

out_text = Path("../results/table_text_lift.csv")
with open(out_text, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["model", "dataset", "readout", "f1_off_mean", "f1_on_mean", "mean_delta", "p", "n_pairs"])
    for model in MODELS:
        for dataset in DATASETS:
            readout = winner_readout(model, dataset)
            if readout is None: w.writerow([model, dataset, "", "", "", "", "", 0]); continue
            off = groups.get((model, dataset, False, readout)); on = groups.get((model, dataset, True, readout))
            if not off or not on: w.writerow([model, dataset, readout, "", "", "", "", 0]); continue
            t = paired_t(on, off)
            w.writerow([model, dataset, readout, f"{summarise(off)['mean']:.4f}", f"{summarise(on)['mean']:.4f}", f"{t['mean_delta']:.4f}", f"{t['p']:.4f}", t['n_pairs']])
print(f"wrote {out_text}")